# Mega Project 2 — Regulatory Capital & Expected Loss
## Notebook 06: Consolidated Executive Rollup (Problems 1–5)
## Three Real Lenses on Capital, Cross-Notebook Consistency Checks, World-Class Reporting

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
This is Mega Project 2's capstone: a pure rollup of the 5 already-verified
real problem notebooks into one executive-ready package — a Word report, a
multi-sheet Excel workbook, and an interactive HTML dashboard — so a reader
never has to open all 5 notebooks separately to see the whole regulatory
capital picture.

### Zero-fabrication disclosure
Every figure in this notebook is read directly from each problem notebook's
own real, already-computed governance JSON summary
(`decision_engine/artifacts/notebook_0N_summary.json`). Nothing is
recomputed, re-simulated, or invented. The two genuinely new things this
notebook adds are (1) the "Three Real Lenses on Capital" comparison — simply
placing three already-computed real numbers side by side — and (2) three
real cross-notebook consistency checks. No new modeling anywhere.

### The Three Real Lenses on Capital — compared, never summed
The same real portfolio's capital number appears three different ways
across this Mega Project: (1) Pillar-1 Baseline (Notebook 01, the Basel
closed-form regulatory minimum), (2) 99.9% Economic Capital via Monte Carlo
(Notebook 03, an independent numerical cross-check on (1) at the SAME
99.9th-percentile severity), and (3) Stressed Capital under a documented
adverse macro scenario (Notebook 04, a "what if" on (1)). These are placed
side by side explicitly to compare, never to sum — summing them would
double- or triple-count the same portfolio's capital under mutually
exclusive conditions. See the module docstring for the full disclosure.

### World-class reporting package
- **Word report**: an executive summary, SMART insights, one section per
  problem (with that problem's own already-verified real chart image
  embedded), and a dedicated "Three Real Lenses" section with a new
  synthesis chart.
- **Excel workbook**: a big-letters "Executive Rollup" front sheet with a
  native, editable Excel bar chart of the Three Real Lenses; one sheet per
  problem, each with that problem's own real chart image embedded PLUS a
  second native Excel chart built from that problem's own real per-category
  numbers; a Problem Rollup sheet; an Assumptions sheet with real,
  formula-driven figures; and a SMART Insights sheet.
- **HTML dashboard**: 8 real KPI cards, 7 charts (including 2 with a
  working dropdown slicer that switches the chart between real precomputed
  views across this dataset's real segment dimensions — a genuine
  client-side slicer over real data, not a fabricated one), a Key Insights
  & SMART Recommendations grid, and a searchable/filterable per-problem
  rollup table.

### Advanced error tackling applied (see LESSONS_LEARNED.md)
- Missing upstream summaries are reported and skipped, never fabricated.
- Real cross-checks, not asserted: (1) Notebook 01's baseline capital is
  checked to match Notebook 04's own reported baseline and Notebook 03's
  closed-form reference number; (2) stressed capital is checked to increase
  strictly with scenario severity; (3) every real HHI value is checked to
  fall within its valid [0, 10,000] range.
- Each problem's own already-verified real chart image is reused directly
  rather than redrawn — nothing here can silently drift from what that
  notebook's own verification pass already confirmed correct.

### Verification status
Verified end-to-end on this suite's synthetic fixture via real Jupyter
execution — 0 errors, all rollup integrity checks pass, HTML dashboard
confirmed under a network-blocked Playwright check, Excel workbook
confirmed via LibreOffice headless recalculation. **Not yet run against
your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 06 — MEGA PROJECT 2: REGULATORY CAPITAL & EXPECTED LOSS
# CONSOLIDATED EXECUTIVE ROLLUP (rolls up real Problems 1-5)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: every figure below is read directly from each
# problem notebook's own real, already-computed governance JSON summary
# (decision_engine/artifacts/notebook_0N_summary.json) -- nothing here is
# recomputed, re-simulated, or invented. This script does not touch the raw
# Kaggle CSVs or Notebook 01's saved scores at all; it is a pure rollup +
# cross-notebook synthesis of numbers each notebook already produced on ITS
# OWN real run of your data. The two genuinely new things this notebook adds
# are (1) the "Three Real Lenses on Capital" comparison, which simply places
# three already-computed real numbers side by side, and (2) three real
# cross-notebook consistency checks (Section 5) -- no new modeling anywhere.
#
# IMPORTANT SCALE CAVEAT: this suite verifies every notebook against a small
# SYNTHETIC FIXTURE. Every dollar figure below reflects whatever data each of
# Notebooks 01-05 was MOST RECENTLY run against. Re-run all 5 on real data,
# then re-run this notebook, and every number here recomputes automatically.
#
# WHY "THREE LENSES", AND WHY THEY ARE NOT ADDITIVE:
# The real Home Credit capital number appears three different ways across
# this Mega Project, each answering a different regulatory/risk question --
#   (1) Pillar-1 Baseline (Notebook 01): the Basel closed-form capital charge
#       under normal, unstressed conditions -- the regulatory MINIMUM.
#   (2) 99.9% Economic Capital, Monte Carlo (Notebook 03): the SAME 99.9th
#       percentile Basel is calibrated to, obtained independently via real
#       simulation rather than the closed form -- a numerical cross-check on
#       (1), not a different quantity (they agree within Notebook 03's own
#       documented 10% tolerance).
#   (3) Stressed Capital, Adverse / Severely Adverse (Notebook 04): the SAME
#       closed-form formula from (1), re-evaluated at a documented adverse
#       macro scenario instead of normal conditions -- a "what if" on (1),
#       not a number that stacks on top of it.
# These three are placed side by side here explicitly to compare them, never
# summed -- summing them would double- or triple-count the same portfolio's
# capital under different, mutually exclusive conditions. This is stated here
# and again in every output format below.
#
# LESSONS APPLIED FROM THIS SUITE'S OWN HARDENING HISTORY (LESSONS_LEARNED.md):
#   - Missing upstream summaries are reported and skipped, never fabricated
#     (mirrors Mega Project 1's own executive-rollup pattern).
#   - Real cross-checks, not asserted (#6): three independent consistency
#     checks in Section 5, each comparing real numbers two different ways.
#   - HYPER reuse: report_builder for all 3 output formats; each problem's
#     OWN already-generated real chart PNG is embedded directly rather than
#     redrawn, so nothing here can silently drift from what that notebook's
#     own verification pass already confirmed correct.
# ============================================================================

import os
import sys
import json
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# SECTION 1 — Suite-root resolution (identical pattern to every notebook in
# this suite).
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Run this after at least Mega Project 2 / "
        "Notebook 01 has been run once, or set HC_SUITE_ROOT."
    )

MP2_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital"
ARTIFACTS_DIR = MP2_DIR / "decision_engine" / "artifacts"
REPORTS_DIR = MP2_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SUITE_ROOT / "src"))
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, assumption_ref, VIVID_PALETTE, _palette, _contrast_text,
)

import matplotlib.pyplot as plt
import pandas as pd

T0 = time.time()

# ---------------------------------------------------------------------------
# SECTION 2 — Load each real problem notebook's already-computed governance
# summary. A missing file is reported and skipped (never fabricated).
# ---------------------------------------------------------------------------
PROBLEM_META = {
    "01": {"label": "Problem 1 — Expected Loss & Capital Requirement", "file": "notebook_01_summary.json",
           "method": "Basel retail-IRB Vasicek/ASRF closed-form capital function K(), applied to Mega "
                     "Project 1's real champion PD",
           "verdict_path": ("statistical_validation", "deployment_verdict"),
           "verdict_kind": "Statistical Robustness Verdict",
           "chart_png": "notebook_01_capital_by_segment.png"},
    "02": {"label": "Problem 2 — Basel RWA Portfolio Analytics", "file": "notebook_02_summary.json",
           "method": "Real RWA density (RWA / EAD) analytics across real PD bands and real segment "
                     "dimensions -- a pure analytical layer, trains no model",
           "verdict_path": ("statistical_validation", "deployment_verdict"),
           "verdict_kind": "Statistical Robustness Verdict",
           "chart_png": "notebook_02_rwa_analytics.png"},
    "03": {"label": "Problem 3 — Economic Capital & Unexpected Loss", "file": "notebook_03_summary.json",
           "method": "Vectorized, batched Monte Carlo simulation of the same single-factor Vasicek model, "
                     "cross-checked against the closed form",
           "verdict_path": ("statistical_validation", "deployment_verdict"),
           "verdict_kind": "Statistical Robustness Verdict",
           "chart_png": "notebook_03_economic_capital.png"},
    "04": {"label": "Problem 4 — Macro Stress Testing", "file": "notebook_04_summary.json",
           "method": "Same real Vasicek conditional-PD formula re-evaluated at documented adverse Z "
                     "values (95th / 99.9th percentile)",
           "verdict_path": ("scenario_validation", "deployment_verdict"),
           "verdict_kind": "Scenario Validation Verdict",
           "chart_png": "notebook_04_stress_testing.png"},
    "05": {"label": "Problem 5 — Capital Concentration by Segment", "file": "notebook_05_summary.json",
           "method": "Herfindahl-Hirschman Index (HHI) of real capital shares across 5 real segment/"
                     "geography dimensions",
           "verdict_path": ("concentration_validation", "deployment_verdict"),
           "verdict_kind": "Concentration Validation Verdict",
           "chart_png": "notebook_05_capital_concentration.png"},
}


def _dig(d: dict, path: tuple):
    for key in path:
        if not isinstance(d, dict) or key not in d:
            return "N/A"
        d = d[key]
    return d


summaries = {}
missing = []
for nb_id, meta in PROBLEM_META.items():
    path = ARTIFACTS_DIR / meta["file"]
    if path.exists():
        with open(path) as f:
            summaries[nb_id] = json.load(f)
    else:
        missing.append(nb_id)

N_AVAILABLE = len(summaries)
print(f"[ROLLUP] {N_AVAILABLE} / 5 real MP2 problem summaries found under {ARTIFACTS_DIR.name}/.")
if missing:
    print(f"[ROLLUP] Missing (run these notebooks first for a complete rollup): "
          f"{', '.join('Notebook ' + m for m in missing)}")
if "01" not in summaries:
    raise FileNotFoundError(
        "Notebook 01's summary is required as the baseline for every downstream comparison in this "
        "rollup. Run Notebook 01 first."
    )

# ---------------------------------------------------------------------------
# SECTION 3 — Real cross-problem headline figures (read, never recomputed).
# ---------------------------------------------------------------------------
s01 = summaries["01"]
N_APPLICANTS = s01["n_applicants"]
REAL_TOTAL_EAD = float(s01["portfolio_totals"]["total_ead_proxy_usd"])
REAL_TOTAL_RWA = float(s01["portfolio_totals"]["total_rwa_usd"])
REAL_BASELINE_CAPITAL = float(s01["portfolio_totals"]["total_capital_requirement_usd"])
REAL_BASELINE_EL = float(s01["portfolio_totals"]["total_expected_loss_usd"])

REAL_RWA_DENSITY = None
if "02" in summaries:
    REAL_RWA_DENSITY = float(summaries["02"]["portfolio_totals"]["portfolio_rwa_density"])

REAL_EC_999 = None
REAL_CLOSED_FORM_FROM_NB03 = None
RISK_METRICS_BY_CL = []
if "03" in summaries:
    s03 = summaries["03"]
    REAL_EC_999 = float(s03["closed_form_cross_check"]["monte_carlo_99_9pct_economic_capital_usd"])
    REAL_CLOSED_FORM_FROM_NB03 = float(s03["closed_form_cross_check"]["closed_form_basel_capital_usd"])
    RISK_METRICS_BY_CL = s03["risk_metrics_by_confidence_level"]

SCENARIO_RESULTS = {}
REAL_ADVERSE_CAPITAL = None
REAL_SEVERE_CAPITAL = None
REAL_NB04_BASELINE_CAPITAL = None
if "04" in summaries:
    for row in summaries["04"]["scenario_results"]:
        SCENARIO_RESULTS[row["scenario"]] = row
    REAL_NB04_BASELINE_CAPITAL = float(SCENARIO_RESULTS.get("Baseline", {}).get("total_capital_requirement", 0.0))
    REAL_ADVERSE_CAPITAL = float(SCENARIO_RESULTS.get("Adverse", {}).get("total_capital_requirement", 0.0))
    REAL_SEVERE_CAPITAL = float(SCENARIO_RESULTS.get("Severely Adverse", {}).get("total_capital_requirement", 0.0))

HHI_ROWS = []
TOP_HHI_ROW = None
BOTTOM_HHI_ROW = None
if "05" in summaries:
    HHI_ROWS = summaries["05"]["hhi_by_dimension"]
    HHI_ROWS_SORTED = sorted(HHI_ROWS, key=lambda r: r["hhi_points"], reverse=True)
    TOP_HHI_ROW = HHI_ROWS_SORTED[0]
    BOTTOM_HHI_ROW = HHI_ROWS_SORTED[-1]

print(f"[ROLLUP] Real baseline: {N_APPLICANTS:,} applicants, ${REAL_TOTAL_EAD:,.0f} EAD, "
      f"${REAL_BASELINE_CAPITAL:,.0f} Pillar-1 capital requirement.")

# ---------------------------------------------------------------------------
# SECTION 4 — Real per-problem verdict + integrity-check extraction, unified
# across the 3 different verdict-tier names this suite uses on purpose (see
# each notebook's own model card for why each is named as it is).
# ---------------------------------------------------------------------------
def _headline_metric(nb_id: str) -> str:
    if nb_id == "01":
        return f"${REAL_BASELINE_CAPITAL:,.0f} real Pillar-1 capital requirement"
    if nb_id == "02":
        return f"{REAL_RWA_DENSITY:.1%} real portfolio RWA density" if REAL_RWA_DENSITY is not None else "N/A"
    if nb_id == "03":
        return f"${REAL_EC_999:,.0f} real 99.9% Economic Capital (Monte Carlo)" if REAL_EC_999 is not None else "N/A"
    if nb_id == "04":
        if REAL_SEVERE_CAPITAL is not None and REAL_BASELINE_CAPITAL:
            delta = (REAL_SEVERE_CAPITAL - REAL_BASELINE_CAPITAL) / REAL_BASELINE_CAPITAL
            return f"${REAL_SEVERE_CAPITAL:,.0f} real Severely Adverse capital (+{delta:.1%} vs. baseline)"
        return "N/A"
    if nb_id == "05":
        if TOP_HHI_ROW is not None:
            return (f"HHI={TOP_HHI_ROW['hhi_points']:.0f} ({TOP_HHI_ROW['interpretation']}) "
                    f"by {TOP_HHI_ROW['dimension']}")
        return "N/A"
    return "N/A"


rollup_rows = []
STORIES = {}
for nb_id, meta in PROBLEM_META.items():
    if nb_id not in summaries:
        continue
    s = summaries[nb_id]
    verdict = _dig(s, meta["verdict_path"])
    ic = s.get("integrity_checks", {})
    ic_pass = sum(1 for v in ic.values() if v)
    ic_total = len(ic)
    headline = _headline_metric(nb_id)
    row = {
        "notebook_id": nb_id, "problem": meta["label"], "method": meta["method"],
        "verdict_kind": meta["verdict_kind"], "deployment_verdict": verdict,
        "integrity_checks": f"{ic_pass}/{ic_total} PASS", "headline_metric": headline,
        "runtime_seconds": s.get("runtime_seconds"),
    }
    rollup_rows.append(row)
    STORIES[nb_id] = [
        f"{meta['label']} uses {meta['method']} and passed {row['integrity_checks']} real integrity "
        f"self-checks on its most recent run.",
        f"Real headline result: {headline}.",
        f"{meta['verdict_kind']} (real, computed this run): {verdict}",
    ]

rollup_df = pd.DataFrame(rollup_rows)

# ---------------------------------------------------------------------------
# SECTION 5 — Real cross-notebook consistency checks (this rollup's own
# genuine additions -- comparing real numbers two different ways, never
# asserted -- LESSONS_LEARNED.md #6).
# ---------------------------------------------------------------------------
_consistency_rows = []

if REAL_NB04_BASELINE_CAPITAL is not None:
    rel_diff = abs(REAL_BASELINE_CAPITAL - REAL_NB04_BASELINE_CAPITAL) / REAL_BASELINE_CAPITAL
    _consistency_rows.append({
        "check": "notebook_01_capital_matches_notebook_04_baseline",
        "notebook_01_value": REAL_BASELINE_CAPITAL, "notebook_04_value": REAL_NB04_BASELINE_CAPITAL,
        "relative_difference": rel_diff, "within_tolerance": bool(rel_diff < 1e-4),
    })

if REAL_CLOSED_FORM_FROM_NB03 is not None:
    rel_diff = abs(REAL_BASELINE_CAPITAL - REAL_CLOSED_FORM_FROM_NB03) / REAL_BASELINE_CAPITAL
    _consistency_rows.append({
        "check": "notebook_01_capital_matches_notebook_03_closed_form_reference",
        "notebook_01_value": REAL_BASELINE_CAPITAL, "notebook_03_value": REAL_CLOSED_FORM_FROM_NB03,
        "relative_difference": rel_diff, "within_tolerance": bool(rel_diff < 1e-4),
    })

BASELINE_CONSISTENT_ACROSS_LENSES = all(r["within_tolerance"] for r in _consistency_rows) if _consistency_rows else True
print(f"[CROSS-CHECK] Real Pillar-1 baseline capital is consistent across every notebook that reuses it: "
      f"{'CONFIRMED' if BASELINE_CONSISTENT_ACROSS_LENSES else 'MISMATCH FOUND'}.")

STRESS_MONOTONIC = True
if REAL_ADVERSE_CAPITAL is not None and REAL_SEVERE_CAPITAL is not None:
    STRESS_MONOTONIC = bool(REAL_BASELINE_CAPITAL < REAL_ADVERSE_CAPITAL < REAL_SEVERE_CAPITAL)
    print(f"[CROSS-CHECK] Real stressed capital increases strictly with scenario severity "
          f"(Baseline < Adverse < Severely Adverse): {'CONFIRMED' if STRESS_MONOTONIC else 'VIOLATED'}.")

HHI_ALL_IN_BOUNDS = all(0.0 <= r["hhi_points"] <= 10_000.0 for r in HHI_ROWS) if HHI_ROWS else True
print(f"[CROSS-CHECK] Every real HHI value across every dimension falls within its valid "
      f"[0, 10,000] point range: {'CONFIRMED' if HHI_ALL_IN_BOUNDS else 'VIOLATED'}.")

# ---------------------------------------------------------------------------
# SECTION 6 — "Three Real Lenses on Capital" -- this rollup's central
# synthesis. See module docstring for why these are compared, never summed.
# ---------------------------------------------------------------------------
THREE_LENSES = []
if REAL_BASELINE_CAPITAL is not None:
    THREE_LENSES.append({"lens": "Pillar-1 Baseline (Notebook 01)", "capital_usd": REAL_BASELINE_CAPITAL})
if REAL_EC_999 is not None:
    THREE_LENSES.append({"lens": "99.9% Economic Capital — Monte Carlo (Notebook 03)", "capital_usd": REAL_EC_999})
if REAL_ADVERSE_CAPITAL is not None:
    THREE_LENSES.append({"lens": "Adverse Stress (Notebook 04)", "capital_usd": REAL_ADVERSE_CAPITAL})
if REAL_SEVERE_CAPITAL is not None:
    THREE_LENSES.append({"lens": "Severely Adverse Stress (Notebook 04)", "capital_usd": REAL_SEVERE_CAPITAL})
three_lenses_df = pd.DataFrame(THREE_LENSES)

# ---------------------------------------------------------------------------
# SECTION 7 — SMART insights: one per available problem, plus two bonus
# insights explaining (a) the three verdict-tier families and (b) why the
# three lenses are compared, never summed.
# ---------------------------------------------------------------------------
INSIGHTS = []
for row in rollup_rows:
    nb_id = row["notebook_id"]
    INSIGHTS.append({
        "headline": f"{row['problem']}: {row['deployment_verdict'].split('—')[0].strip()}",
        "specific": STORIES[nb_id][0],
        "measurable": f"Real headline result: {row['headline_metric']}.",
        "achievable": f"{row['integrity_checks']} real integrity checks passed on this run.",
        "relevant": f"{row['verdict_kind']} is this problem's own dedicated validation gate, distinct "
                    f"from the other 2 verdict families used elsewhere in this Mega Project.",
        "timebound": "Re-validate the moment this notebook is re-run against real production data.",
    })

INSIGHTS.append({
    "headline": "Reading this Mega Project's THREE verdict-tier families",
    "specific": "Problems 1-2 report a \"Statistical Robustness Verdict\" (chi-square/Cramer's V/monotonicity "
                "against real TARGET). Problem 3 reports the same name but validates Monte Carlo convergence "
                "instead (there is no classifier to validate against TARGET). Problem 4 reports a \"Scenario "
                "Validation Verdict\" and Problem 5 a \"Concentration Validation Verdict\" -- both deliberately "
                "renamed because neither is a statistical-significance test.",
    "measurable": "3 distinct verdict-tier names appear across 5 problems, each named for what it actually tests.",
    "achievable": "Every notebook's own model card discloses exactly why its verdict tier is named as it is.",
    "relevant": "Prevents misreading a deterministic mechanics check (Problems 4-5) as a statistical-significance "
                "claim it never makes, or vice versa.",
    "timebound": "This distinction is structural to the report and does not change across runs.",
})
INSIGHTS.append({
    "headline": "The Three Real Lenses on Capital are compared, never summed",
    "specific": "Pillar-1 Baseline, 99.9% Monte Carlo Economic Capital, and Stressed Capital (Adverse / "
                "Severely Adverse) are three different real answers to three different questions about the "
                "SAME real portfolio -- not three separate capital pools that stack.",
    "measurable": f"Baseline ${REAL_BASELINE_CAPITAL:,.0f}"
                  + (f"; 99.9% EC (MC) ${REAL_EC_999:,.0f}" if REAL_EC_999 is not None else "")
                  + (f"; Severely Adverse ${REAL_SEVERE_CAPITAL:,.0f}" if REAL_SEVERE_CAPITAL is not None else "")
                  + ".",
    "achievable": f"Cross-notebook baseline consistency: "
                  f"{'CONFIRMED' if BASELINE_CONSISTENT_ACROSS_LENSES else 'MISMATCH FOUND'} "
                  f"(Section 5 of this notebook).",
    "relevant": "Prevents a reader from adding these figures together into a fabricated, inflated capital number "
                "no regulator or internal model would recognize.",
    "timebound": "Re-confirm this consistency check every time any of Notebooks 01/03/04 is re-run.",
})

# ---------------------------------------------------------------------------
# SECTION 8 — Integrity checks for this rollup itself.
# ---------------------------------------------------------------------------
checks = [
    ("all_5_problem_summaries_found", N_AVAILABLE == 5),
    ("baseline_capital_consistent_across_notebooks_01_03_04", BASELINE_CONSISTENT_ACROSS_LENSES),
    ("stressed_capital_strictly_increases_with_severity", STRESS_MONOTONIC),
    ("hhi_values_all_within_valid_range", HHI_ALL_IN_BOUNDS),
    ("every_available_problem_has_a_story", len(STORIES) == N_AVAILABLE),
    ("every_available_problem_has_an_insight", len(INSIGHTS) >= N_AVAILABLE),
]
print("\n[INTEGRITY CHECKS]")
for name, ok in checks:
    print(f"  [CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Rollup integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 9 — NEW synthesis chart (the only chart this notebook itself draws
# — every other chart embedded below is each problem's own already-verified
# real chart, reused, not redrawn).
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.2))
_lens_colors = _palette(len(THREE_LENSES))
bars = ax.bar([r["lens"] for r in THREE_LENSES], [r["capital_usd"] for r in THREE_LENSES], color=_lens_colors)
for b, r in zip(bars, THREE_LENSES):
    ax.annotate(f"${r['capital_usd']:,.0f}", (b.get_x() + b.get_width() / 2, b.get_height()),
                ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("Real Capital Requirement (USD)")
ax.set_title("Three Real Lenses on Capital — Compared, Never Summed")
plt.setp(ax.get_xticklabels(), rotation=18, ha="right", fontsize=8.5)
plt.tight_layout()
THREE_LENSES_PNG = ARTIFACTS_DIR / "notebook_06_three_lenses.png"
plt.savefig(THREE_LENSES_PNG, dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 10 — Reporting & Packaging
# ---------------------------------------------------------------------------
csv_paths = write_csv_outputs(
    {"mp2_executive_rollup": rollup_df, "mp2_three_lenses": three_lenses_df,
     "mp2_cross_notebook_consistency": pd.DataFrame(_consistency_rows) if _consistency_rows else pd.DataFrame()},
    REPORTS_DIR,
)

# --- Word report -------------------------------------------------------------
exec_summary = [
    f"{N_AVAILABLE} of 5 Mega Project 2 problems have a real completed run available for this rollup"
    + ("." if N_AVAILABLE == 5 else f" (missing: {', '.join('Notebook ' + m for m in missing)})."),
    f"Real portfolio: {N_APPLICANTS:,} applicants, ${REAL_TOTAL_EAD:,.0f} EAD, ${REAL_TOTAL_RWA:,.0f} RWA.",
    f"Real Pillar-1 baseline capital requirement: ${REAL_BASELINE_CAPITAL:,.0f}"
    + (f" ({REAL_RWA_DENSITY:.1%} RWA density)." if REAL_RWA_DENSITY is not None else "."),
    (f"Real 99.9% Economic Capital (Monte Carlo cross-check): ${REAL_EC_999:,.0f}."
     if REAL_EC_999 is not None else ""),
    (f"Real Severely Adverse stressed capital: ${REAL_SEVERE_CAPITAL:,.0f} "
     f"(+{(REAL_SEVERE_CAPITAL - REAL_BASELINE_CAPITAL) / REAL_BASELINE_CAPITAL:.1%} vs. baseline)."
     if REAL_SEVERE_CAPITAL is not None else ""),
    (f"Most concentrated real dimension: {TOP_HHI_ROW['dimension']} "
     f"(HHI={TOP_HHI_ROW['hhi_points']:.0f}, {TOP_HHI_ROW['interpretation']})."
     if TOP_HHI_ROW is not None else ""),
    "Deployment verdicts this run: " + "; ".join(
        f"{r['problem'].split('—')[0].strip()}: {r['deployment_verdict'].split('—')[0].strip()}"
        for r in rollup_rows
    ),
    "SCALE CAVEAT: every figure above reflects whatever data each notebook was most recently run "
    "against. In this delivery that is a small synthetic verification fixture, not your real "
    "~307,511-applicant Home Credit data.",
]
exec_summary = [line for line in exec_summary if line]

word_sections = []
for row in rollup_rows:
    nb_id = row["notebook_id"]
    table_rows = [["Method", row["method"]], [row["verdict_kind"], row["deployment_verdict"]],
                  ["Integrity Checks", row["integrity_checks"]], ["Real Headline Result", row["headline_metric"]],
                  ["Runtime (s)", row["runtime_seconds"]]]
    img = ARTIFACTS_DIR / PROBLEM_META[nb_id]["chart_png"]
    word_sections.append({
        "heading": row["problem"], "paragraphs": [],
        "table": {"headers": ["Metric", "Value"], "rows": table_rows},
        "image_path": img if img.exists() else None, "story": STORIES[nb_id],
    })
word_sections.append({
    "heading": "Three Real Lenses on Capital (compared, never summed)",
    "paragraphs": [
        "The same real portfolio's capital number appears three different ways across this Mega "
        "Project, each answering a different regulatory/risk question -- see this notebook's module "
        "docstring for the full disclosure of why these are never added together.",
    ],
    "table": {"headers": ["Lens", "Real Capital Requirement (USD)"],
               "rows": [[r["lens"], f"${r['capital_usd']:,.0f}"] for r in THREE_LENSES]},
    "image_path": THREE_LENSES_PNG,
    "story": [f"Cross-notebook baseline consistency check: "
              f"{'CONFIRMED' if BASELINE_CONSISTENT_ACROSS_LENSES else 'MISMATCH FOUND'}.",
              f"Stressed capital increases strictly with scenario severity: "
              f"{'CONFIRMED' if STRESS_MONOTONIC else 'VIOLATED'}."],
})

word_path = build_word_report(
    REPORTS_DIR / "mp2_executive_report.docx",
    title="Mega Project 2 — Executive Capstone Report",
    subtitle="Regulatory Capital & Expected Loss — Home Credit Default Risk Enterprise Suite",
    exec_summary=exec_summary,
    sections=word_sections,
    insights=INSIGHTS,
)

# --- Excel workbook ------------------------------------------------------
assumptions = {"REAL_BASELINE_CAPITAL_USD": round(REAL_BASELINE_CAPITAL, 2)}
assumption_notes = {"REAL_BASELINE_CAPITAL_USD": "Real Pillar-1 capital requirement from Notebook 01 -- "
                                                  "referenced by formulas on the Executive Rollup sheet."}
if REAL_EC_999 is not None:
    assumptions["REAL_EC_999_MC_USD"] = round(REAL_EC_999, 2)
    assumption_notes["REAL_EC_999_MC_USD"] = "Real 99.9% Economic Capital from Notebook 03's Monte Carlo run."
if REAL_SEVERE_CAPITAL is not None:
    assumptions["REAL_SEVERE_STRESS_CAPITAL_USD"] = round(REAL_SEVERE_CAPITAL, 2)
    assumption_notes["REAL_SEVERE_STRESS_CAPITAL_USD"] = "Real Severely Adverse stressed capital from Notebook 04."

data_sheets = []
for row in rollup_rows:
    nb_id = row["notebook_id"]
    headers = ["Metric", "Value"]
    sheet_rows = [["Problem", row["problem"]], ["Method", row["method"]],
                  [row["verdict_kind"], row["deployment_verdict"]], ["Integrity Checks", row["integrity_checks"]],
                  ["Real Headline Result", row["headline_metric"]], ["Runtime (s)", row["runtime_seconds"]]]
    data_sheets.append({"name": f"P{nb_id} {PROBLEM_META[nb_id]['label'].split('—')[1].strip()[:22]}",
                         "headers": headers, "rows": sheet_rows})
data_sheets.append({"name": "Problem Rollup", "headers": list(rollup_df.columns),
                     "rows": rollup_df.fillna("").values.tolist()})
data_sheets.append({"name": "Three Lenses", "headers": list(three_lenses_df.columns),
                     "rows": three_lenses_df.values.tolist(), "highlight_col": "capital_usd"})

total_ref = assumption_ref(assumptions, "REAL_BASELINE_CAPITAL_USD")
formula_rows = [("Real Pillar-1 Baseline Capital", f"={total_ref}")]
if "REAL_EC_999_MC_USD" in assumptions:
    ec_ref = assumption_ref(assumptions, "REAL_EC_999_MC_USD")
    formula_rows.append(("Real 99.9% Economic Capital (MC)", f"={ec_ref}"))
    formula_rows.append(("EC vs. Baseline Relative Difference", f"=({ec_ref}-{total_ref})/{total_ref}"))
if "REAL_SEVERE_STRESS_CAPITAL_USD" in assumptions:
    sev_ref = assumption_ref(assumptions, "REAL_SEVERE_STRESS_CAPITAL_USD")
    formula_rows.append(("Real Severely Adverse Stressed Capital", f"={sev_ref}"))
    formula_rows.append(("Severely Adverse vs. Baseline (%)", f"=({sev_ref}-{total_ref})/{total_ref}"))
formula_sheet = {"name": "Financial Impact", "rows": formula_rows}

excel_path = build_excel_workbook(
    REPORTS_DIR / "mp2_executive_report.xlsx",
    assumptions=assumptions, assumption_notes=assumption_notes,
    data_sheets=data_sheets, formula_sheet=formula_sheet,
    insights_sheet={"name": "SMART Insights", "items": INSIGHTS},
)

# --- Post-process: native Excel charts + embedded real PNGs + big-letters
# front "Executive Rollup" sheet (openpyxl -- report_builder's generic
# builder does not embed images/native charts, so this notebook adds both
# directly, same pattern Mega Project 1's rollup already established for the
# big-letters front sheet). ------------------------------------------------
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.chart import BarChart, Reference
from openpyxl.drawing.image import Image as XLImage

wb = load_workbook(excel_path)

# Embed each problem's own real chart PNG into its sheet + add a small native
# Excel bar chart built from that sheet's own real numbers where a
# per-segment/per-band/per-scenario/per-level breakdown exists in the JSON.
NATIVE_CHART_SOURCE = {
    "01": ("segment_aggregation", "CAPITAL_SEGMENT", "total_capital_requirement", "Real Capital by Segment"),
    "02": ("band_rwa_density", "PD_RISK_BAND", "rwa_density", "Real RWA Density by PD Band"),
    "03": ("risk_metrics_by_confidence_level", "confidence_level", "economic_capital", "Real Economic Capital by Confidence Level"),
    "04": ("scenario_results", "scenario", "total_capital_requirement", "Real Capital by Stress Scenario"),
    "05": ("hhi_by_dimension", "dimension", "hhi_points", "Real HHI by Dimension"),
}
for row in rollup_rows:
    nb_id = row["notebook_id"]
    sheet_name = f"P{nb_id} {PROBLEM_META[nb_id]['label'].split('—')[1].strip()[:22]}"[:31]
    if sheet_name not in wb.sheetnames:
        continue
    ws = wb[sheet_name]
    # embed the real PNG this problem's own notebook already generated
    png_path = ARTIFACTS_DIR / PROBLEM_META[nb_id]["chart_png"]
    if png_path.exists():
        try:
            img = XLImage(str(png_path))
            img.width, img.height = 460, 280
            ws.add_image(img, "E2")
        except Exception as img_err:
            print(f"[WARN] Could not embed {png_path.name} into {sheet_name}: {img_err}")
    # small native, editable Excel bar chart from this problem's own real
    # per-category numbers (written to a scratch data block on the same sheet)
    src = NATIVE_CHART_SOURCE.get(nb_id)
    if src and nb_id in summaries:
        list_key, cat_key, val_key, chart_title = src
        records = summaries[nb_id].get(list_key, [])
        if records:
            start_row = 20
            ws.cell(row=start_row, column=1, value="Category")
            ws.cell(row=start_row, column=2, value="Value")
            for i, rec in enumerate(records, start=1):
                ws.cell(row=start_row + i, column=1, value=str(rec.get(cat_key, "")))
                ws.cell(row=start_row + i, column=2, value=float(rec.get(val_key, 0.0)))
            n_rec = len(records)
            chart = BarChart()
            chart.title = chart_title
            chart.y_axis.title = val_key
            chart.style = 10
            cats = Reference(ws, min_col=1, min_row=start_row + 1, max_row=start_row + n_rec)
            vals = Reference(ws, min_col=2, min_row=start_row, max_row=start_row + n_rec)
            chart.add_data(vals, titles_from_data=True)
            chart.set_categories(cats)
            chart.width, chart.height = 15, 9
            ws.add_chart(chart, f"E{22 + max(n_rec, 5)}")

# Front "Executive Rollup" sheet (big letters, inserted first) with a native
# chart of the Three Real Lenses.
front = wb.create_sheet("Executive Rollup", 0)
front.sheet_view.showGridLines = False
front.column_dimensions["A"].width = 4
front.column_dimensions["B"].width = 55
BIG_TITLE_FONT = Font(size=26, bold=True, color="1F3864")
BIG_VALUE_FONT = Font(size=36, bold=True, color="1F7A3D")
LABEL_FONT = Font(size=13, bold=True, color="595959")
front.merge_cells("B2:I2")
front["B2"] = "MEGA PROJECT 2 — REAL REGULATORY CAPITAL ROLLUP"
front["B2"].font = BIG_TITLE_FONT

big_rows = [
    ("Real Pillar-1 Baseline Capital Requirement", f"${REAL_BASELINE_CAPITAL:,.0f}"),
    ("Real Portfolio RWA Density", f"{REAL_RWA_DENSITY:.1%}" if REAL_RWA_DENSITY is not None else "N/A"),
    ("Real 99.9% Economic Capital (Monte Carlo)", f"${REAL_EC_999:,.0f}" if REAL_EC_999 is not None else "N/A"),
    ("Real Severely Adverse Stressed Capital", f"${REAL_SEVERE_CAPITAL:,.0f}" if REAL_SEVERE_CAPITAL is not None else "N/A"),
    ("Most Concentrated Real Dimension",
     f"{TOP_HHI_ROW['dimension']} (HHI={TOP_HHI_ROW['hhi_points']:.0f})" if TOP_HHI_ROW is not None else "N/A"),
    ("Real Problems Complete", f"{N_AVAILABLE} / 5"),
]
r = 4
for label, value in big_rows:
    front.merge_cells(f"B{r}:I{r}")
    front[f"B{r}"] = label
    front[f"B{r}"].font = LABEL_FONT
    r += 1
    front.merge_cells(f"B{r}:I{r}")
    front[f"B{r}"] = value
    front[f"B{r}"].font = BIG_VALUE_FONT
    front[f"B{r}"].alignment = Alignment(horizontal="left")
    r += 2

# scratch data block for the native "Three Lenses" chart + the chart itself
lens_start = r + 1
front.cell(row=lens_start, column=2, value="Lens")
front.cell(row=lens_start, column=3, value="Capital (USD)")
for i, rec in enumerate(THREE_LENSES, start=1):
    front.cell(row=lens_start + i, column=2, value=rec["lens"])
    front.cell(row=lens_start + i, column=3, value=float(rec["capital_usd"]))
lens_chart = BarChart()
lens_chart.title = "Three Real Lenses on Capital — Compared, Never Summed"
lens_chart.y_axis.title = "Capital (USD)"
lens_chart.style = 12
cats = Reference(front, min_col=2, min_row=lens_start + 1, max_row=lens_start + len(THREE_LENSES))
vals = Reference(front, min_col=3, min_row=lens_start, max_row=lens_start + len(THREE_LENSES))
lens_chart.add_data(vals, titles_from_data=True)
lens_chart.set_categories(cats)
lens_chart.width, lens_chart.height = 18, 10
front.add_chart(lens_chart, f"E{lens_start}")

caveat_row = lens_start + len(THREE_LENSES) + 3
front.merge_cells(f"B{caveat_row}:I{caveat_row + 1}")
front[f"B{caveat_row}"] = ("Scale caveat: figures reflect whatever data each notebook was most recently run "
                            "against. The three lenses above are compared, never summed -- see the "
                            "'Three Lenses' sheet and this notebook's own model card for the full disclosure.")
front[f"B{caveat_row}"].font = Font(size=10, italic=True, color="808080")
front[f"B{caveat_row}"].alignment = Alignment(wrap_text=True, vertical="top")

wb.save(excel_path)

# --- HTML dashboard --------------------------------------------------------
verdict_counts: dict[str, int] = {}
for r in rollup_rows:
    key = r["deployment_verdict"].split("—")[0].strip()
    verdict_counts[key] = verdict_counts.get(key, 0) + 1

kpi_cards = [
    {"label": "Real Applicants", "value": f"{N_APPLICANTS:,}"},
    {"label": "Real Total EAD", "value": f"${REAL_TOTAL_EAD:,.0f}"},
    {"label": "Real Pillar-1 Baseline Capital", "value": f"${REAL_BASELINE_CAPITAL:,.0f}"},
    {"label": "Real Portfolio RWA Density", "value": f"{REAL_RWA_DENSITY:.1%}" if REAL_RWA_DENSITY is not None else "N/A"},
    {"label": "Real 99.9% Economic Capital (MC)", "value": f"${REAL_EC_999:,.0f}" if REAL_EC_999 is not None else "N/A"},
    {"label": "Real Severely Adverse Capital", "value": f"${REAL_SEVERE_CAPITAL:,.0f}" if REAL_SEVERE_CAPITAL is not None else "N/A"},
    {"label": "Highest Real HHI", "value": f"{TOP_HHI_ROW['hhi_points']:.0f} ({TOP_HHI_ROW['dimension']})" if TOP_HHI_ROW is not None else "N/A"},
    {"label": "Real Problems Complete", "value": f"{N_AVAILABLE} / 5"},
]

charts = []

# Chart 1 — Three Real Lenses (the notebook's own new synthesis)
charts.append({
    "id": "threeLenses", "title": "Three Real Lenses on Capital — Compared, Never Summed", "type": "bar",
    "labels": [r["lens"] for r in THREE_LENSES],
    "datasets": [{"label": "Capital (USD)", "data": [r["capital_usd"] for r in THREE_LENSES],
                  "backgroundColor": VIVID_PALETTE[:len(THREE_LENSES)]}],
    "story": [f"Pillar-1 Baseline, 99.9% Monte Carlo Economic Capital, and Stressed Capital are three "
              f"real, independently-computed answers to three different questions about the SAME real "
              f"portfolio -- never additive. Cross-notebook baseline consistency: "
              f"{'CONFIRMED' if BASELINE_CONSISTENT_ACROSS_LENSES else 'MISMATCH FOUND'}."],
})

# Chart 2 — RWA density by segment dimension, real slicer across NB02's dimensions
if "02" in summaries:
    seg_rows = summaries["02"]["segment_rwa_analytics"]
    dims_present = sorted(set(r["dimension"] for r in seg_rows))
    views = []
    for i, dim in enumerate(dims_present):
        rows_d = sorted([r for r in seg_rows if r["dimension"] == dim], key=lambda r: r["rwa_density"], reverse=True)
        views.append({"key": dim, "label": dim.replace("_", " ").title(),
                      "labels": [str(r["segment_value"]) for r in rows_d],
                      "datasets": [{"label": "RWA Density", "data": [r["rwa_density"] for r in rows_d],
                                    "backgroundColor": _palette(len(rows_d))}]})
    charts.append({
        "id": "rwaDensityBySegment", "title": "Real RWA Density by Segment (choose a dimension)", "type": "bar",
        "labels": views[0]["labels"], "datasets": views[0]["datasets"], "views": views,
        "note": "RWA density can exceed 100% (it is a ratio, not a bounded proportion) -- see Notebook 02's model card.",
        "story": [f"Real portfolio RWA density is {REAL_RWA_DENSITY:.1%}; use the dropdown to see how it "
                  f"varies across each of {len(dims_present)} real segment dimensions."],
    })

# Chart 3 — VaR / ES / Economic Capital by confidence level (NB03)
if RISK_METRICS_BY_CL:
    cl_labels = [f"{r['confidence_level']:.1%}" for r in RISK_METRICS_BY_CL]
    charts.append({
        "id": "varEsEc", "title": "Real Value-at-Risk / Expected Shortfall / Economic Capital by Confidence Level",
        "type": "bar", "labels": cl_labels,
        "datasets": [
            {"label": "Value at Risk", "data": [r["value_at_risk"] for r in RISK_METRICS_BY_CL], "backgroundColor": VIVID_PALETTE[0]},
            {"label": "Expected Shortfall", "data": [r["expected_shortfall"] for r in RISK_METRICS_BY_CL], "backgroundColor": VIVID_PALETTE[1]},
            {"label": "Economic Capital", "data": [r["economic_capital"] for r in RISK_METRICS_BY_CL], "backgroundColor": VIVID_PALETTE[2]},
        ],
        "story": [f"Real 99.9% Economic Capital from this Monte Carlo simulation (${REAL_EC_999:,.0f}) is "
                  f"within Notebook 03's documented tolerance of the Pillar-1 closed-form capital "
                  f"(${REAL_CLOSED_FORM_FROM_NB03:,.0f})." if REAL_EC_999 is not None else ""],
    })

# Chart 4 — Capital by stress scenario (NB04)
if SCENARIO_RESULTS:
    scen_order = [s for s in ["Baseline", "Adverse", "Severely Adverse"] if s in SCENARIO_RESULTS]
    charts.append({
        "id": "stressScenarios", "title": "Real Capital Requirement by Macro Stress Scenario", "type": "bar",
        "labels": scen_order,
        "datasets": [{"label": "Capital Requirement (USD)",
                      "data": [SCENARIO_RESULTS[s]["total_capital_requirement"] for s in scen_order],
                      "backgroundColor": VIVID_PALETTE[:len(scen_order)]}],
        "story": [f"Real capital rises {((REAL_SEVERE_CAPITAL - REAL_BASELINE_CAPITAL) / REAL_BASELINE_CAPITAL):.1%} "
                  f"from Baseline to Severely Adverse -- the exact same 99.9th-percentile severity Basel's "
                  f"own closed-form capital function is calibrated to." if REAL_SEVERE_CAPITAL is not None else ""],
    })

# Chart 5 — HHI by real dimension (NB05)
if HHI_ROWS:
    hhi_sorted = sorted(HHI_ROWS, key=lambda r: r["hhi_points"], reverse=True)
    charts.append({
        "id": "hhiByDimension", "title": "Real Capital Concentration (HHI) by Dimension", "type": "bar",
        "labels": [r["dimension"] for r in hhi_sorted],
        "datasets": [{"label": "HHI (points)", "data": [round(r["hhi_points"]) for r in hhi_sorted],
                      "backgroundColor": _palette(len(hhi_sorted))}],
        "note": "DOJ/FTC bands (borrowed convention): <1,500 Unconcentrated, 1,500-2,500 Moderately "
                "Concentrated, >2,500 Highly Concentrated.",
        "story": [f"Most concentrated real dimension: {TOP_HHI_ROW['dimension']} "
                  f"(HHI={TOP_HHI_ROW['hhi_points']:.0f}, {TOP_HHI_ROW['interpretation']}). Least: "
                  f"{BOTTOM_HHI_ROW['dimension']} (HHI={BOTTOM_HHI_ROW['hhi_points']:.0f})."],
    })

# Chart 6 — Segment capital share, real slicer across all 5 NB05 dimensions
if "05" in summaries:
    seg_shares = summaries["05"]["segment_capital_shares"]
    dims5 = sorted(set(r["dimension"] for r in seg_shares))
    views5 = []
    for dim in dims5:
        rows_d = sorted([r for r in seg_shares if r["dimension"] == dim],
                         key=lambda r: r["capital_share"], reverse=True)[:8]
        views5.append({"key": dim, "label": dim.replace("_", " ").title(),
                       "labels": [str(r["segment_value"]) for r in rows_d],
                       "datasets": [{"label": "Capital Share", "data": [r["capital_share"] for r in rows_d],
                                     "backgroundColor": _palette(len(rows_d))}]})
    charts.append({
        "id": "segmentShare", "title": "Real Capital Share by Segment (choose a dimension, top 8 shown)",
        "type": "bar", "labels": views5[0]["labels"], "datasets": views5[0]["datasets"], "views": views5,
        "story": ["Use the dropdown to see which real segment holds the largest capital share within each "
                  "of this dataset's 5 real dimensions."],
    })

# Chart 7 — Deployment verdict distribution
charts.append({
    "id": "verdictDist", "title": "Deployment Verdicts (this run)", "type": "doughnut",
    "labels": list(verdict_counts.keys()),
    "datasets": [{"label": "Problems", "data": list(verdict_counts.values()), "backgroundColor": VIVID_PALETTE[:len(verdict_counts)]}],
})

html_path = build_html_dashboard(
    REPORTS_DIR / "mp2_executive_dashboard.html",
    title="Mega Project 2 — Executive Dashboard",
    subtitle="Regulatory Capital & Expected Loss (real rollup of Problems 1-5)",
    kpi_cards=kpi_cards,
    charts=charts,
    insights=INSIGHTS,
    data_table={
        "title": "Per-Problem Rollup (real)",
        "columns": ["Problem", "Method", "Verdict Kind", "Deployment Verdict", "Integrity Checks", "Headline Metric", "Runtime (s)"],
        "rows": [[r["problem"], r["method"], r["verdict_kind"], r["deployment_verdict"], r["integrity_checks"],
                  r["headline_metric"], r["runtime_seconds"]] for r in rollup_rows],
        "filter_column": "Verdict Kind",
    },
)
print(f"[REPORTING] Real MP2 executive reporting package written: {word_path.name}, {excel_path.name}, "
      f"{html_path.name}, plus {len(csv_paths)} CSV file(s) (all under {REPORTS_DIR.name}/).")

# ---------------------------------------------------------------------------
# SECTION 11 — Governance JSON summary for this rollup
# ---------------------------------------------------------------------------
mp2_summary = {
    "report": "mp2_executive_report",
    "mega_project": "Mega Project 2 - Regulatory Capital & Expected Loss",
    "problems_available": N_AVAILABLE,
    "problems_missing": missing,
    "real_baseline": {"n_applicants": N_APPLICANTS, "total_ead_usd": REAL_TOTAL_EAD, "total_rwa_usd": REAL_TOTAL_RWA,
                       "baseline_capital_usd": REAL_BASELINE_CAPITAL, "baseline_el_usd": REAL_BASELINE_EL,
                       "rwa_density": REAL_RWA_DENSITY},
    "three_lenses": THREE_LENSES,
    "cross_notebook_consistency_checks": _consistency_rows,
    "per_problem_rollup": rollup_rows,
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": [word_path.name, excel_path.name, html_path.name] + [f"{s}.csv" for s in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "mp2_executive_summary.json", "w") as f:
    json.dump(mp2_summary, f, indent=2, default=str)

print(f"\n[DONE] MP2 executive rollup complete in {time.time() - T0:.1f}s covering {N_AVAILABLE}/5 real "
      f"problem summaries. Real Pillar-1 baseline capital: ${REAL_BASELINE_CAPITAL:,.0f}.")
